# Przetwarzanie danych

In [12]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
sns.set(style="whitegrid")

import warnings
warnings.filterwarnings("ignore")
df = pd.read_csv('../data/eda_data.csv')
df.head()

,Unnamed: 0,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status,arrival_fulldate,no_of_people
0,3871,INN03872,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,Canceled,2017-07-01,2
1,26642,INN26643,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,Canceled,2017-07-01,2
2,16946,INN16947,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,Canceled,2017-07-01,2
3,3898,INN03899,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,Canceled,2017-07-01,2
4,5884,INN05885,1,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,80.0,0,Not_Canceled,2017-07-01,1


In [13]:
df = df.drop(columns=['Unnamed: 0','Booking_ID' ]
)
df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status,arrival_fulldate,no_of_people
0,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,Canceled,2017-07-01,2
1,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,Canceled,2017-07-01,2
2,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,Canceled,2017-07-01,2
3,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,Canceled,2017-07-01,2
4,1,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,80.0,0,Not_Canceled,2017-07-01,1


In [14]:
df['booking_status']=df['booking_status'].map({'Canceled':1, 'Not_Canceled':0})
df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status,arrival_fulldate,no_of_people
0,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,1,2017-07-01,2
1,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,1,2017-07-01,2
2,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,101.5,0,1,2017-07-01,2
3,2,0,0,2,Meal Plan 2,0,Room_Type 1,257,Online,0,0,0,101.5,0,1,2017-07-01,2
4,1,0,0,2,Meal Plan 2,0,Room_Type 1,257,Offline,0,0,0,80.0,0,0,2017-07-01,1


In [15]:
X = df.drop("booking_status", axis=1)
y = df["booking_status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=123)

In [16]:
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Wszystkie cechy:", X_train.columns.tolist())
print("Numeryczne cechy:", num_features)
print("Kategoryczne cechy:", cat_features)

Wszystkie cechy: ['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space', 'room_type_reserved', 'lead_time', 'market_segment_type', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'arrival_fulldate', 'no_of_people']
Numeryczne cechy: ['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'required_car_parking_space', 'lead_time', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'no_of_people']
Kategoryczne cechy: ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type', 'arrival_fulldate']


In [17]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

# Dopasowanie pipeline do danych treningowych
data_pipeline.fit(X_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['no_of_adults',
                                                   'no_of_children',
                                                   'no_of_weekend_nights',
                                                   'no_of_week_nights',
                                                   'required_car_parking_space',
                                                   'lead_time',
                                                   'repeated_guest',
                                                   'no_of_previous_cancellations',
                                                   'no_of_previous_bookings_not_canceled',
                                                   'avg_price_per_room',
                                                   'no_of_special_requests',
                                                   'no_of_people']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['type_of_meal_plan',
                                                   'room_type_reserved',
                                                   'market_segment_type',
                                                   'arrival_fulldate'])]))])

In [18]:
feature_names = (
    pd.Index(data_pipeline.named_steps['preprocessor'].get_feature_names_out())
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

X_train_processed = pd.DataFrame(data_pipeline.transform(X_train).toarray(),
                                columns=feature_names)

X_test_processed = pd.DataFrame(data_pipeline.transform(X_test).toarray(),
                                columns=feature_names)

X_valid_processed = pd.DataFrame(data_pipeline.transform(X_valid).toarray(),
                                columns=feature_names)
X_train_processed.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,...,arrival_fulldate_2018-12-22,arrival_fulldate_2018-12-23,arrival_fulldate_2018-12-24,arrival_fulldate_2018-12-25,arrival_fulldate_2018-12-26,arrival_fulldate_2018-12-27,arrival_fulldate_2018-12-28,arrival_fulldate_2018-12-29,arrival_fulldate_2018-12-30,arrival_fulldate_2018-12-31
0,0.295836,-0.263211,-0.933138,0.560895,-0.175773,-0.106124,-0.158465,-0.064414,-0.087869,0.498338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.295836,-0.263211,0.214283,0.560895,-0.175773,1.458094,-0.158465,-0.064414,-0.087869,-0.871470,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.295836,-0.263211,1.361703,-0.147357,-0.175773,-0.456322,-0.158465,-0.064414,-0.087869,-1.030783,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.295836,-0.263211,-0.933138,-0.147357,-0.175773,0.103995,-0.158465,-0.064414,-0.087869,-0.508748,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.295836,-0.263211,-0.933138,-0.855610,-0.175773,-0.934925,-0.158465,-0.064414,-0.087869,-2.941119,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
X_train_processed.to_csv('../data/X_train.csv',index=False)
X_test_processed.to_csv('../data/X_test.csv',index=False)
X_valid_processed.to_csv('../data/X_valid.csv',index=False)
y_train.to_csv('../data/y_train.csv',index=False)
y_test.to_csv('../data/y_test.csv',index=False)
y_valid.to_csv('../data/y_valid.csv',index=False)